In [28]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import matplotlib as mpl

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używane urządzenie: {device}")

Używane urządzenie: cpu


In [25]:
class DuffingNet(nn.Module):
    def __init__(self, input_dim=8, output_dim=1):
        super(DuffingNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.GELU(),
            nn.Linear(64, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        return self.net(x)

model = DuffingNet().to(device)

model_path = "duffing_test.pt"

try:
    state_dict = torch.load(model_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    print("Wagi modelu zostały załadowane pomyślnie!")
    
    model.eval()
    
except FileNotFoundError:
    print(f"Błąd: Nie znaleziono pliku '{model_path}'. Sprawdź ścieżkę.")
except Exception as e:
    print(f"Wystąpił błąd podczas ładowania modelu: {e}")

Wagi modelu zostały załadowane pomyślnie!


In [ ]:
# Parametry
alpha, beta, delta, gamma, omega = -1.0, 1.0, 0.3, 0.5, 1.2
x0, v0 = 1.0, 0.0

T_MAX = 30.0
NUM_POINTS = 10000

# Obliczenia modelu PINN
t_real = torch.linspace(0, T_MAX, NUM_POINTS, device=device, requires_grad=True).unsqueeze(1)
t_net = t_real / T_MAX

params = torch.tensor([[alpha, beta, delta, gamma, omega, x0, v0]], device=device)
params_expanded = params.expand(NUM_POINTS, -1)

model_input = torch.cat([params_expanded, t_net], dim=1)

model.eval()
x_pred = model(model_input)

v_pred = torch.autograd.grad(
    outputs=x_pred,
    inputs=t_real,
    grad_outputs=torch.ones_like(x_pred),
    create_graph=False
)[0]

# Konwersja na tablice numpy używane w obu wykresach
x_plot = x_pred.detach().cpu().numpy().flatten()
v_plot = v_pred.detach().cpu().numpy().flatten()

In [3]:
plt.figure(figsize=(10, 8))
plt.plot(x_plot, v_plot, lw=0.5, color='#3F8FE0', alpha=0.8, label='Trajektoria')
plt.title(f"Przestrzeń fazowa\n$\\alpha={alpha}, \\beta={beta}, \\delta={delta}, \\gamma={gamma}, \\omega={omega}$", pad=15)
plt.xlabel('$x(t)$')
plt.ylabel('$v(t) = \\frac{dx}{dt}$')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

NameError: name 'plt' is not defined

In [ ]:
mpl.rcParams['animation.embed_limit'] = 50.0


fig, ax = plt.subplots(figsize=(10, 8))

padding_x = (x_plot.max() - x_plot.min()) * 0.05
padding_v = (v_plot.max() - v_plot.min()) * 0.05
ax.set_xlim(x_plot.min() - padding_x, x_plot.max() + padding_x)
ax.set_ylim(v_plot.min() - padding_v, v_plot.max() + padding_v)

ax.set_title(f"Animacja trajektorii w przestrzeni fazowej\n$\\alpha={alpha}, \\beta={beta}, \\delta={delta}, \\gamma={gamma}, \\omega={omega}$", pad=15)
ax.set_xlabel('$x(t)$')
ax.set_ylabel('$v(t) = \\frac{dx}{dt}$')
ax.grid(True, alpha=0.3)

trace_line, = ax.plot([], [], lw=1.0, color='#3F8FE0', alpha=0.8, label='Ślad trajektorii')
head_point, = ax.plot([], [], 'o', color='#D63A82', markersize=6, label='Aktualny stan', zorder=5)
ax.legend()

STEP = 20
frames = range(0, NUM_POINTS, STEP)

def update(i):
    trace_line.set_data(x_plot[:i], v_plot[:i])
    head_point.set_data([x_plot[i]], [v_plot[i]])
    return trace_line, head_point

ani = animation.FuncAnimation(fig, update, frames=frames, interval=20, blit=True)

plt.close(fig)
HTML(ani.to_jshtml())
